<a href="https://colab.research.google.com/github/harrisonm23-byte/HuggingFaceML1/blob/claude%2Fhuggingface-ml-research-setup-ulvgve/notebooks/12b_whats_saved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12b — What notebook 12 has saved so far

Shows, for every run and version, which of the five days are saved on your Google Drive, and which day notebook 12 will run next. **No GPU needed**, so it works even when Colab's GPU limit is reached.

**To run:** Runtime → Run all, then allow Google Drive access with the **same Google account** you used for notebook 12. ✓ = day saved, · = not yet.

In [ ]:
# Connect Google Drive (choose the same Google account you used for notebook 12)
import os, json, glob, datetime
from google.colab import drive
drive.mount("/content/drive")
MY_DRIVE = "/content/drive/MyDrive"

In [ ]:
# The days notebook 12 runs, in its order (the try-out is 2 runs: the full takeover and one line)
DAYS = {"2011-01-07": "7 Jan", "2011-01-12": "12 Jan", "2011-04-01": "1 Apr", "2011-06-08": "8 Jun", "2011-08-05": "5 Aug"}
VERSIONS = ["real", "loss", "gain", "loss + you", "gain + you", "no ask"]
MODES = ["takeover 5 days", "takeover 1 day"]
RUNS = 2                                  # as in notebook 12 (10 for the final round)

# Every saved days.jsonl on this Drive: the study folder first, then up to two folders down from My Drive
found = sorted(set(glob.glob(f"{MY_DRIVE}/silver_study/**/days.jsonl", recursive=True) + glob.glob(f"{MY_DRIVE}/days.jsonl")
                   + glob.glob(f"{MY_DRIVE}/*/days.jsonl") + glob.glob(f"{MY_DRIVE}/*/*/days.jsonl")))
if not found:
    print("Nothing saved on this Google account's Drive (no days.jsonl anywhere).")
    print("If notebook 12 showed lines like '1/60 | real | ...', it saved to another Google account or only inside Colab.")

for path in found:
    rows, cut = [], 0
    for line in open(path).read().splitlines():
        try:
            rows.append(json.loads(line))
        except ValueError:                # a line cut off by a disconnect: notebook 12 redoes that day
            cut += 1
    saved = {(r["run"], r["version"], r["date"]) for r in rows}
    folder = os.path.dirname(path)
    acts = len(glob.glob(os.path.join(folder, "acts", "*.npz")))
    when = datetime.datetime.fromtimestamp(os.path.getmtime(path), datetime.timezone.utc).strftime("%d %b %H:%M")
    total = RUNS * len(VERSIONS) * len(DAYS)
    print(path.replace(MY_DRIVE, "My Drive"))
    print(f"{len(saved)} of {total} days saved | last saved {when} UTC | {acts} activation files | "
          f"model {', '.join(sorted({r.get('model', '?') for r in rows}))}" + (f" | {cut} cut-off line, redone next time" if cut else "")
          + (f" | {len(rows) - len(saved)} day(s) saved twice (two tabs running at once)" if len(rows) > len(saved) else ""))
    print("\n" + f"{'run':<22}{'version':<12}" + "".join(f"{d:>8}" for d in DAYS.values()))
    upnext = None
    for k in range(RUNS):
        for v in VERSIONS:
            marks = ""
            for date in DAYS:
                marks += f"{'✓' if (k, v, date) in saved else '·':>8}"
                if (k, v, date) not in saved and upnext is None:
                    upnext = f"run {k + 1} ({MODES[k % len(MODES)]}) | {v} | {DAYS[date]}"
            print(f"{str(k + 1) + ': ' + MODES[k % len(MODES)]:<22}{v:<12}{marks}")
    print("\nNext to run:", upnext or "nothing, all done: send Claude days.jsonl from this folder", "\n")